In [1]:
from os.path import basename, exists, join, splitext
from os import makedirs
import json
import pandas as pd
from natsort import natsorted

In [2]:
#!pip install natsort

In [3]:
from natsort import natsorted

Function for extracting aCompCor components

In [4]:
def extract_compcor(confounds_df, confounds_meta,
                    n_comps=5, method='tCompCor',
                    tissue=None):

    # Check that we sensible number of components
    assert n_comps > 0

    # Check that method is specified correctly
    assert method in ['aCompCor', 'tCompCor']

    # Check that tissue is specified for aCompCor
    if method == 'aCompCor' and tissue not in ['combined', 'CSF', 'WM']:
        raise AssertionError("Must specify a tissue type "
                             "(combined, CSF, or WM) for aCompCor")

    # Ignore tissue if specified for tCompCor
    if method == 'tCompCor' and tissue:
        print("Warning: tCompCor is not restricted to a tissue "
              f"mask - ignoring tissue specification ({tissue})")
        tissue = None

    # Get CompCor metadata for relevant method
    compcor_meta = {c: confounds_meta[c] for c in confounds_meta
                    if confounds_meta[c]['Method'] == method
                    and confounds_meta[c]['Retained']}

    # If aCompCor, filter metadata for tissue mask
    if method == 'aCompCor':
        compcor_meta = {c: compcor_meta[c] for c in compcor_meta
                        if compcor_meta[c]['Mask'] == tissue}

    # Make sure metadata components are sorted properly
    comp_sorted = natsorted(compcor_meta)
    for i, comp in enumerate(comp_sorted):
        if comp != comp_sorted[-1]:
            comp_next = comp_sorted[i + 1]
            assert (compcor_meta[comp]['SingularValue'] >
                    compcor_meta[comp_next]['SingularValue'])

    # Either get top n components
    if n_comps >= 1.0:
        n_comps = int(n_comps)
        if len(comp_sorted) >= n_comps:
            comp_selector = comp_sorted[:n_comps]
        else:
            comp_selector = comp_sorted
            print(f"Warning: Only {len(comp_sorted)} {method} "
                  f"components available ({n_comps} requested)")

    # Or components necessary to capture n proportion of variance
    else:
        comp_selector = []
        for comp in comp_sorted:
            comp_selector.append(comp)
            if (compcor_meta[comp]['CumulativeVarianceExplained']
                > n_comps):
                break

    # Check we didn't end up with degenerate 0 components
    assert len(comp_selector) > 0

    # Grab the actual component time series
    confounds_compcor = confounds_df[comp_selector]
    return confounds_compcor

Function for extracting group of (variable number) confounds

In [5]:
def extract_group(confounds_df, groups):
    
    # Expect list, so change if string
    if type(groups) == str:
        groups = [groups]
    
    # Filter for all columns with label
    confounds_group = []
    for group in groups:
        group_cols = [col for col in confounds_df.columns
                      if group in col]
        confounds_group.append(confounds_df[group_cols])
    confounds_group = pd.concat(confounds_group, axis=1)
    
    return confounds_group

Function for loading in confounds files

In [6]:
def load_confounds(confounds_fn):

    # Load the confounds TSV files
    confounds_df = pd.read_csv(confounds_fn, sep='\t')

    # Load the JSON sidecar metadata
    with open(splitext(confounds_fn)[0] + '.json') as f:
        confounds_meta = json.load(f)
    return confounds_df, confounds_meta

In [7]:
def load_confounds(confounds_fn):

    # Load the confounds TSV files
    confounds_df = pd.read_csv(confounds_fn +".tsv", sep='\t')

    # Load the JSON sidecar metadata
    with open(confounds_fn + '.json') as f:
        confounds_meta = json.load(f)
    return confounds_df, confounds_meta

Function for extracting confounds (including CompCor)

In [8]:
def extract_confounds(confounds_df, confounds_meta, model_spec):

    # Pop out confound groups of variable number
    groups = set(model_spec['confounds']).intersection(
                    ['cosine', 'motion_outlier'])

    # Grab the requested confounds
    confounds = confounds_df[[c for c in model_spec['confounds']
                              if c not in groups]]
    
    # Grab confound groups if present
    if groups:
        confounds_group = extract_group(confounds_df,
                                        groups)
        confounds = pd.concat([confounds, confounds_group],
                              axis=1)

    # Get aCompCor / tCompCor confounds if requested
    compcors = set(model_spec).intersection(
                    ['aCompCor', 'tCompCor'])
    if compcors:
        for compcor in compcors:
            if type(model_spec[compcor]) == dict:
                model_spec[compcor] = [model_spec[compcor]]
            for compcor_kws in model_spec[compcor]:
                confounds_compcor = extract_compcor(
                    confounds_df,
                    confounds_meta,
                    method=compcor,
                    **compcor_kws)
                confounds = pd.concat([confounds,
                                       confounds_compcor],
                                      axis=1)
    return confounds

In [9]:
data_dir = "/jukebox/graziano/coolCatIsaac/ATM/data/bids/derivatives/fmriprep/"
mnirois_dir = "/jukebox/graziano/coolCatIsaac/ATM/data/work/rois/"
behav_p = '/jukebox/graziano/coolCatIsaac/ATM/data/behavioral'
sav_work = "/jukebox/graziano/coolCatIsaac/ATM/data/work/results/"

In [10]:
import os

In [11]:
sub_list = ["sub-000","sub-001","sub-003","sub-004","sub-005","sub-006","sub-007","sub-008","sub-009",
            "sub-010","sub-011","sub-012","sub-013","sub-014","sub-015", "sub-016","sub-017", 
            "sub-018", "sub-019", "sub-020","sub-021"]
sub_list = ["sub-021"]
sub_list = ["sub-000","sub-001","sub-003"]
sub_list = ["sub-002"]
sub_list = ["sub-004","sub-005","sub-006","sub-007","sub-008","sub-009","sub-010","sub-011","sub-012"
           ,"sub-013","sub-014"]
sub_list = ["sub-015", "sub-016","sub-017", "sub-018", "sub-019", "sub-020","sub-021", "sub-021"]
sub_list = ["sub-021","sub-022","sub-023","sub-024","sub-025","sub-026","sub-027"]
sub_list = ["sub-026","sub-027"]
sub_list = ["sub-021"]


In [13]:
################## My TRY ##################
# check runs
for sub in sub_list:
    for run in range(1,11):
        file1 = os.path.join(data_dir, sub + "/ses-01/func")
        confounds_fn = os.path.join(data_dir, sub + "/ses-01/func","%s_ses-01_task-Attn_run-%s_desc-confounds_timeseries" % (sub, run))
        #x, y = load_confounds(file_in)
        #print(file_in)
        #base_dir = '/jukebox/hasson/snastase/narratives'

        # Set an AFNI pipeline output directory (either -smooth or -nosmooth)
        afni_pipe = 'afni-head_mot'
        afni_dir = join(data_dir, afni_pipe)

        model =  {'confounds':
                  ['trans_x', 'trans_y', 'trans_z',
                   'rot_x', 'rot_y', 'rot_z', 'cosine'],
                  'aCompCor': [{'n_comps': 5, 'tissue': 'CSF'},
                               {'n_comps': 5, 'tissue': 'WM'}]}

        # Loop through tasks and subjects and grab confound files

        # Make directory if it doesn't exist
        ort_dir = join(afni_dir, sub, 'func')
        if not exists(ort_dir):
            makedirs(ort_dir)

        # Grab confound files for multiple runs if present
        #confounds_fns = natsorted(
        #    task_meta[task][subject]['confounds'])

        # Loop through confound files (in case of multiple runs)
        #for confounds_fn in confounds_fns:
        confounds_df, confounds_meta = load_confounds(confounds_fn)

        # Extract confounds based on model
        confounds = extract_confounds(confounds_df,
                                      confounds_meta,
                                      model)
        
        # Create output 1D file for AFNI and save
        ort_1d = splitext(basename(confounds_fn).replace(
            'desc-confounds',
            f'desc-model'))[0] + '.1D'
        ort_fn = join(ort_dir, ort_1d)
        #confounds.to_csv(ort_fn, sep='\t', header=False,
          #               index=False)
        
        # Also create CSVs with headers for convenience
        ort_csv = splitext(basename(confounds_fn).replace(
            'desc-confounds',
            f'desc-model'))[0] + '.csv'
        ort_fn = join(ort_dir, ort_csv)
        #confounds.to_csv(ort_fn, sep=',', index=False)

        print(f"Assembled confound models for {sub}")

Assembled confound models for sub-021
Assembled confound models for sub-021
Assembled confound models for sub-021
Assembled confound models for sub-021
Assembled confound models for sub-021
Assembled confound models for sub-021
Assembled confound models for sub-021
Assembled confound models for sub-021
Assembled confound models for sub-021
Assembled confound models for sub-021


In [17]:
confounds

,trans_x,trans_y,trans_z,rot_x,rot_y,rot_z,cosine00,cosine01,cosine02,a_comp_cor_00,a_comp_cor_01,a_comp_cor_02,a_comp_cor_03,a_comp_cor_04,a_comp_cor_10,a_comp_cor_11,a_comp_cor_12,a_comp_cor_13,a_comp_cor_14
0,0.009831,-0.082387,0.096656,0.002536,0.001597,-0.000600,0.097820,0.097812,0.097798,-0.240324,0.028324,-0.234798,0.104291,-0.179017,0.111997,-0.095482,0.032961,0.098615,-0.014996
1,0.016454,-0.081711,0.122619,0.003040,0.001324,-0.000600,0.097798,0.097724,0.097599,-0.205672,-0.191260,-0.068118,-0.062966,-0.081523,0.161957,-0.139983,0.087861,0.126883,-0.099772
2,0.019090,-0.060929,0.145420,0.002665,0.001228,-0.000506,0.097754,0.097547,0.097202,-0.144629,-0.198675,0.004089,0.039598,-0.139753,0.156104,-0.147274,0.060884,0.094064,-0.197210
3,0.019779,-0.063263,0.153707,0.001947,0.001115,-0.000677,0.097688,0.097282,0.096607,-0.073477,-0.136174,0.009482,0.049011,-0.067966,0.117593,-0.146742,0.018763,0.104558,-0.255323
4,-0.006669,0.015851,0.172245,-0.001511,0.000541,-0.000867,0.097599,0.096929,0.095816,0.011050,-0.038505,0.012961,0.134654,-0.041064,0.087682,-0.167762,-0.063076,-0.018730,-0.203140
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,-0.082859,0.123999,-0.520136,0.003629,0.002264,0.001472,-0.097599,0.096929,-0.095816,-0.085202,0.175587,-0.085058,0.044448,0.130241,0.084916,0.034681,0.029682,-0.088313,0.082073
205,-0.082881,0.004445,-0.510620,0.004157,0.002264,0.001521,-0.097688,0.097282,-0.096607,-0.124416,0.057782,-0.094592,0.063553,0.102250,0.111913,0.025834,0.017726,-0.012441,0.116960
206,-0.082693,0.100386,-0.522230,0.004376,0.002643,0.001602,-0.097754,0.097547,-0.097202,-0.079584,-0.047514,-0.085975,0.016138,0.052614,0.076378,0.029606,0.032936,0.014756,0.081695
207,-0.105239,0.043272,-0.524871,0.005871,0.004368,0.001037,-0.097798,0.097724,-0.097599,-0.053097,-0.177866,0.053041,0.007291,0.039953,0.026738,-0.075082,0.036142,0.083936,0.079602


In [2]:
model =  {'confounds':
          ['trans_x', 'trans_y', 'trans_z',
           'rot_x', 'rot_y', 'rot_z', 'cosine'],
          'aCompCor': [{'n_comps': 5, 'tissue': 'CSF'},
                       {'n_comps': 5, 'tissue': 'WM'}]}

In [3]:
model

{'confounds': ['trans_x',
  'trans_y',
  'trans_z',
  'rot_x',
  'rot_y',
  'rot_z',
  'cosine'],
 'aCompCor': [{'n_comps': 5, 'tissue': 'CSF'}, {'n_comps': 5, 'tissue': 'WM'}]}